# Merge Intervals

## What Is It?
The Merge Intervals pattern deals with problems involving **overlapping ranges or intervals**. You work with pairs `[start, end]` and need to combine, count, or manipulate them efficiently.

---

## Core Strategy
1. **Sort by start time**
2. Iterate, checking if current interval overlaps with the previous
3. **Overlap condition**: `current.start <= previous.end`
4. If overlap: **extend** `previous.end = max(previous.end, current.end)`
5. If no overlap: push the previous and start tracking the current

---

## Four Overlap Cases (ASCII Diagram)
```
prev:  |-------|         prev:  |-------|         prev:  |-------|         prev:  |-------|
curr:      |-------|     curr:  |---|              curr:     |---|          curr:  |--------|
  Case 1: partial       Case 2: contained in      Case 3: partial         Case 4: curr contains
  overlap right         prev (left side)           overlap left            prev entirely
```
All 4 cases merge to: `[min(start1,start2), max(end1,end2)]`.

**Non-overlap**: `curr.start > prev.end` → no merge, add prev to result.

---

## Templates

### Template 1 — Merge Intervals
```python
def merge(intervals):
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:          # overlap
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])     # no overlap
    return merged
```

### Template 2 — Insert Interval
```python
def insert(intervals, newInterval):
    result = []
    i = 0
    # Add all intervals that end before new one starts
    while i < len(intervals) and intervals[i][1] < newInterval[0]:
        result.append(intervals[i]); i += 1
    # Merge all overlapping intervals
    while i < len(intervals) and intervals[i][0] <= newInterval[1]:
        newInterval[0] = min(newInterval[0], intervals[i][0])
        newInterval[1] = max(newInterval[1], intervals[i][1])
        i += 1
    result.append(newInterval)
    result.extend(intervals[i:])
    return result
```

### Template 3 — Interval Intersection
```python
def intersect(A, B):
    i = j = 0
    result = []
    while i < len(A) and j < len(B):
        lo = max(A[i][0], B[j][0])
        hi = min(A[i][1], B[j][1])
        if lo <= hi: result.append([lo, hi])
        if A[i][1] < B[j][1]: i += 1
        else: j += 1
    return result
```

---

## Complexity
| Operation | Time | Space |
|-----------|------|-------|
| Sort + merge | O(n log n) | O(n) |
| Insert (sorted) | O(n) | O(n) |
| Intersection (two sorted) | O(m+n) | O(m+n) |
| Min rooms (events line sweep) | O(n log n) | O(n) |

---

## Key Tips
- **Always sort by start** unless the input is already sorted
- For minimum rooms/groups: use a **min-heap** of end times or a **sweep line** (events)
- For "remove minimum intervals to make non-overlapping": sort by **end time** (greedy)
- Difference array / prefix sum trick works when you need cumulative overlap counts

---
# Easy Problems (20)

### 1. Merge Intervals (LC 56) — Basic

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Given an array of intervals, merge all overlapping intervals.

### Approach
Sort by start. Iterate and extend the last merged interval if overlap is found.

- **Time**: O(n log n)
- **Space**: O(n)

In [ ]:
def merge(intervals):
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

assert merge([[1,3],[2,6],[8,10],[15,18]]) == [[1,6],[8,10],[15,18]]
assert merge([[1,4],[4,5]]) == [[1,5]]
assert merge([[1,4],[2,3]]) == [[1,4]]
assert merge([[1,2]]) == [[1,2]]
print("All tests passed!")

### 2. Summary Ranges (LC 228)

> 🏢 **Asked by:** Amazon, Google
Given a sorted unique integer array, return the smallest sorted list of ranges that cover all numbers exactly.

### Approach
Track the start of each consecutive run. When a gap appears, close the current range and start a new one.

- **Time**: O(n)
- **Space**: O(n)

In [ ]:
def summaryRanges(nums):
    if not nums:
        return []
    result = []
    start = nums[0]
    for i in range(1, len(nums)):
        if nums[i] != nums[i-1] + 1:
            if start == nums[i-1]:
                result.append(str(start))
            else:
                result.append(f"{start}->{nums[i-1]}")
            start = nums[i]
    # handle the last range
    if start == nums[-1]:
        result.append(str(start))
    else:
        result.append(f"{start}->{nums[-1]}")
    return result

assert summaryRanges([0,1,2,4,5,7]) == ["0->2","4->5","7"]
assert summaryRanges([0,2,3,4,6,8,9]) == ["0","2->4","6","8->9"]
assert summaryRanges([]) == []
assert summaryRanges([1]) == ["1"]
print("All tests passed!")

### 3. Meeting Rooms (LC 252)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Given meeting time intervals, determine if a person can attend all meetings (no overlap).

### Approach
Sort by start. Check if any meeting starts before the previous one ends.

- **Time**: O(n log n)
- **Space**: O(1)

In [ ]:
def canAttendMeetings(intervals):
    intervals.sort(key=lambda x: x[0])
    for i in range(1, len(intervals)):
        if intervals[i][0] < intervals[i-1][1]:
            return False
    return True

assert canAttendMeetings([[0,30],[5,10],[15,20]]) == False
assert canAttendMeetings([[7,10],[2,4]]) == True
assert canAttendMeetings([]) == True
assert canAttendMeetings([[1,5],[5,10]]) == True  # back to back is fine
print("All tests passed!")

### 4. Determine if Two Events Have Conflict (LC 2446)

> 🏢 **Asked by:** Amazon, Google
Two events are given as `["HH:MM", "HH:MM"]`. Return True if they have a time conflict.

### Approach
Two intervals `[s1, e1]` and `[s2, e2]` overlap if and only if `s1 <= e2 AND s2 <= e1`. String comparison works directly for HH:MM format.

- **Time**: O(1)
- **Space**: O(1)

In [ ]:
def haveConflict(event1, event2):
    return event1[0] <= event2[1] and event2[0] <= event1[1]

assert haveConflict(["01:15","02:00"],["02:00","03:00"]) == True
assert haveConflict(["01:00","02:00"],["01:20","03:00"]) == True
assert haveConflict(["10:00","11:00"],["14:00","15:00"]) == False
assert haveConflict(["09:00","10:00"],["10:00","11:00"]) == True
print("All tests passed!")

### 5. Maximum Population Year (LC 1854)

> 🏢 **Asked by:** Amazon, Google
Given `logs[i] = [birth, death]`, return the year with the maximum population (a person is alive in year y if birth <= y < death).

### Approach
Difference array / line sweep. Increment at birth year, decrement at death year. Compute prefix sums to find the year with maximum count.

- **Time**: O(n + range)
- **Space**: O(range)

In [ ]:
def maximumPopulation(logs):
    diff = [0] * 101  # years 1950..2050 mapped to 0..100
    for birth, death in logs:
        diff[birth - 1950] += 1
        diff[death - 1950] -= 1  # person is NOT alive in death year
    
    max_pop, curr_pop, best_year = 0, 0, 1950
    for i, delta in enumerate(diff):
        curr_pop += delta
        if curr_pop > max_pop:
            max_pop = curr_pop
            best_year = 1950 + i
    return best_year

assert maximumPopulation([[1993,1999],[2000,2010]]) == 1993
assert maximumPopulation([[1950,1961],[1960,1971],[1970,1981]]) == 1960
assert maximumPopulation([[1950,2000],[1975,2005],[1950,2000],[1975,2005]]) in [1975, 1950]
print("All tests passed!")

### 6. Points That Intersect With Cars (LC 2848)

> 🏢 **Asked by:** Amazon, Google
Given a list of `[start, end]` intervals representing car positions, return the number of integer points covered by at least one car.

### Approach
Merge overlapping intervals first, then sum the length of each merged interval (end - start + 1 for inclusive ranges).

- **Time**: O(n log n)
- **Space**: O(n)

In [ ]:
def numberOfPoints(nums):
    nums.sort()
    merged = [nums[0][:]]
    for s, e in nums[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return sum(e - s + 1 for s, e in merged)

assert numberOfPoints([[3,6],[1,5],[4,7]]) == 7
assert numberOfPoints([[1,3],[5,8]]) == 7
assert numberOfPoints([[1,2],[3,4],[5,6]]) == 6
assert numberOfPoints([[1,1],[2,2]]) == 2
print("All tests passed!")

### 7. Minimum Number of Moves to Seat Everyone (LC 2037)

> 🏢 **Asked by:** Amazon, Google
Given `seats[]` and `students[]` positions, find the minimum total moves so each student occupies a seat (one per seat).

### Approach
Sort both arrays. Match the i-th student to the i-th seat (sorted order minimizes total distance). Sum absolute differences.

- **Time**: O(n log n)
- **Space**: O(1)

In [ ]:
def minMovesToSeat(seats, students):
    seats.sort()
    students.sort()
    return sum(abs(s - t) for s, t in zip(seats, students))

assert minMovesToSeat([3,1,5],[2,7,4]) == 4
assert minMovesToSeat([4,1,5,9],[1,3,2,6]) == 7
assert minMovesToSeat([2,2,6,6],[1,3,2,6]) == 4
print("All tests passed!")

### 8. Minimum Time Visiting All Points (LC 1266)

> 🏢 **Asked by:** Amazon, Google
Return the minimum time to visit all points in order (can move diagonally, horizontally, or vertically 1 unit per second).

### Approach
From one point to the next, the time is `max(|dx|, |dy|)` — the Chebyshev distance. Diagonals handle both dimensions simultaneously.

- **Time**: O(n)
- **Space**: O(1)

In [ ]:
def minTimeToVisitAllPoints(points):
    total = 0
    for i in range(1, len(points)):
        dx = abs(points[i][0] - points[i-1][0])
        dy = abs(points[i][1] - points[i-1][1])
        total += max(dx, dy)
    return total

assert minTimeToVisitAllPoints([[1,1],[3,4],[-1,0]]) == 7
assert minTimeToVisitAllPoints([[3,2],[-2,2]]) == 5
assert minTimeToVisitAllPoints([[0,0],[1,1],[2,2]]) == 2
print("All tests passed!")

### 9. Range Sum Query - Immutable (LC 303)

> 🏢 **Asked by:** Amazon, Google
Design a structure to answer multiple `sumRange(left, right)` queries on a fixed array.

### Approach
Precompute prefix sums. `sumRange(l, r) = prefix[r+1] - prefix[l]`.

- **Time**: O(n) build, O(1) query
- **Space**: O(n)

In [ ]:
class NumArray:
    def __init__(self, nums):
        self.prefix = [0] * (len(nums) + 1)
        for i, n in enumerate(nums):
            self.prefix[i+1] = self.prefix[i] + n
    
    def sumRange(self, left, right):
        return self.prefix[right+1] - self.prefix[left]

na = NumArray([-2,0,3,-5,2,-1])
assert na.sumRange(0,2) == 1
assert na.sumRange(2,5) == -1
assert na.sumRange(0,5) == -3
print("All tests passed!")

### 10. Count Ways to Group Overlapping Ranges (LC 2580)

> 🏢 **Asked by:** Amazon, Google
Given ranges, count the number of ways to split them into two groups such that no two overlapping ranges are in different groups.

### Approach
Sort by start. Merge ranges into connected components. Each component can go to either group independently. The answer is `2^(number of components) mod 1e9+7`.

- **Time**: O(n log n)
- **Space**: O(1)

In [ ]:
def countWays(ranges):
    MOD = 10**9 + 7
    ranges.sort()
    components = 0
    max_end = -1
    for start, end in ranges:
        if start > max_end:  # new component
            components += 1
        max_end = max(max_end, end)
    return pow(2, components, MOD)

assert countWays([[6,10],[5,15]]) == 2
assert countWays([[1,3],[10,20],[2,5],[4,8]]) == 4
assert countWays([[1,2],[3,4],[5,6]]) == 8
print("All tests passed!")

### 11. Non-overlapping Intervals — Easy Count (LC 435 simplified)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Given intervals, return the minimum number of intervals to remove to make the rest non-overlapping.

### Approach
Sort by end time. Greedily keep intervals with earliest end (greedy activity selection). Count discarded intervals.

- **Time**: O(n log n)
- **Space**: O(1)

In [ ]:
def eraseOverlapIntervals(intervals):
    if not intervals:
        return 0
    intervals.sort(key=lambda x: x[1])
    removed = 0
    last_end = intervals[0][1]
    for i in range(1, len(intervals)):
        if intervals[i][0] < last_end:  # overlap → remove current
            removed += 1
        else:
            last_end = intervals[i][1]
    return removed

assert eraseOverlapIntervals([[1,2],[2,3],[3,4],[1,3]]) == 1
assert eraseOverlapIntervals([[1,2],[1,2],[1,2]]) == 2
assert eraseOverlapIntervals([[1,2],[2,3]]) == 0
assert eraseOverlapIntervals([[1,100],[11,22],[1,11],[2,12]]) == 2
print("All tests passed!")

### 12. Partition Array into Disjoint Intervals (LC 915)

> 🏢 **Asked by:** Google, Amazon
Find the smallest index to partition the array into two parts where every element in the left part is <= every element in the right part.

### Approach
Track the max of left part and the global max so far. The partition point is where the max of left equals the running max — meaning no right element is smaller than left max.

- **Time**: O(n)
- **Space**: O(1)

In [ ]:
def partitionDisjoint(nums):
    left_max = nums[0]
    curr_max = nums[0]
    partition = 0
    for i in range(1, len(nums)):
        if nums[i] < left_max:
            partition = i
            left_max = curr_max
        else:
            curr_max = max(curr_max, nums[i])
    return partition + 1

assert partitionDisjoint([5,0,3,8,6]) == 3
assert partitionDisjoint([1,1,1,0,6,12]) == 4
assert partitionDisjoint([1,2,3,4]) == 1
print("All tests passed!")

### 13. Missing Ranges (LC 163 simplified)

> 🏢 **Asked by:** Amazon, Google
Given a sorted array and a range `[lower, upper]`, return the missing ranges.

### Approach
Process gaps between consecutive values (including boundaries). For each gap > 1, record the missing range.

- **Time**: O(n)
- **Space**: O(n)

In [ ]:
def findMissingRanges(nums, lower, upper):
    result = []
    prev = lower - 1
    for i in range(len(nums) + 1):
        curr = nums[i] if i < len(nums) else upper + 1
        if curr - prev == 2:
            result.append([prev + 1, prev + 1])
        elif curr - prev > 2:
            result.append([prev + 1, curr - 1])
        prev = curr
    return result

assert findMissingRanges([0,1,3,50,75], 0, 99) == [[2,2],[4,49],[51,74],[76,99]]
assert findMissingRanges([], 1, 1) == [[1,1]]
assert findMissingRanges([], -3, -1) == [[-3,-1]]
assert findMissingRanges([-1], -1, -1) == []
print("All tests passed!")

### 14. Check if All 1's Are at Least K Places Away (LC 1437)

> 🏢 **Asked by:** Amazon, Google
Given binary array `nums` and integer `k`, return True if every 1 is at least `k` places from every other 1.

### Approach
Track the last position of a 1. When we see a new 1, check if the gap is at least k.

- **Time**: O(n)
- **Space**: O(1)

In [ ]:
def kLengthApart(nums, k):
    last = -1
    for i, n in enumerate(nums):
        if n == 1:
            if last != -1 and i - last - 1 < k:
                return False
            last = i
    return True

assert kLengthApart([1,0,0,0,1,0,0,1], 2) == True
assert kLengthApart([1,0,0,1,0,1], 2) == False
assert kLengthApart([1,1,1,1,1], 0) == True
assert kLengthApart([0,1,0,1], 1) == True
print("All tests passed!")

### 15. Count Days Spent Together (LC 2409)

> 🏢 **Asked by:** Amazon, Google
Two people travel with arrival/departure dates. Count the days they are both present.

### Approach
Find the intersection of two date intervals: `[max(a_arrive, b_arrive), min(a_depart, b_depart)]`. Convert dates to day-of-year for arithmetic.

- **Time**: O(1)
- **Space**: O(1)

In [ ]:
def countDaysTogether(arriveAlice, leaveAlice, arriveBob, leaveBob):
    days_in_month = [31,28,31,30,31,30,31,31,30,31,30,31]
    
    def toDayOfYear(date):
        month, day = int(date[:2]), int(date[3:])
        return sum(days_in_month[:month-1]) + day
    
    start = max(toDayOfYear(arriveAlice), toDayOfYear(arriveBob))
    end = min(toDayOfYear(leaveAlice), toDayOfYear(leaveBob))
    return max(0, end - start + 1)

assert countDaysTogether("08-15","08-18","08-16","08-19") == 3
assert countDaysTogether("10-01","10-31","11-01","12-31") == 0
assert countDaysTogether("01-01","12-31","01-01","12-31") == 365
print("All tests passed!")

### 16. Number of Rectangles That Can Form The Largest Square (LC 1725)

> 🏢 **Asked by:** Amazon, Google
Given rectangles `[li, wi]`, find the max square side length and count how many rectangles can form it.

### Approach
The max square side from a rectangle is `min(l, w)`. Find the global max, then count rectangles achieving it.

- **Time**: O(n)
- **Space**: O(1)

In [ ]:
def countGoodRectangles(rectangles):
    max_side = max(min(l, w) for l, w in rectangles)
    return sum(1 for l, w in rectangles if min(l, w) == max_side)

assert countGoodRectangles([[5,8],[3,9],[5,12],[16,5]]) == 3
assert countGoodRectangles([[2,3],[3,7],[4,3],[3,7]]) == 3
assert countGoodRectangles([[1,1],[1,1],[1,1]]) == 3
print("All tests passed!")

### 17. Find Right Interval — Simplified

> 🏢 **Asked by:** Amazon, Google
For each interval find the interval with the smallest start point that is >= end of current interval.

### Approach
Build a sorted list of start points with original indices. For each interval's end, binary search to find the first start >= end.

- **Time**: O(n log n)
- **Space**: O(n)

In [ ]:
import bisect

def findRightIntervalSimple(intervals):
    sorted_starts = sorted((iv[0], i) for i, iv in enumerate(intervals))
    keys = [s[0] for s in sorted_starts]
    result = []
    for start, end in intervals:
        pos = bisect.bisect_left(keys, end)
        result.append(sorted_starts[pos][1] if pos < len(keys) else -1)
    return result

assert findRightIntervalSimple([[1,2]]) == [-1]
assert findRightIntervalSimple([[3,4],[2,3],[1,2]]) == [-1,0,1]
assert findRightIntervalSimple([[1,4],[2,3],[3,4]]) == [-1,2,-1]
print("All tests passed!")

### 18. Count Subarrays with Fixed Bounds (LC 2444 simplified)

> 🏢 **Asked by:** Amazon, Google
Count subarrays where min == minK and max == maxK.

### Approach
Slide a window. Track the most recent positions of minK, maxK, and any "bad" element (outside [minK, maxK]). For each right index, valid subarrays start from the position after the last bad element, up to `min(last_min, last_max)`.

- **Time**: O(n)
- **Space**: O(1)

In [ ]:
def countSubarrays(nums, minK, maxK):
    count = 0
    last_min = last_max = bad = -1
    for i, num in enumerate(nums):
        if num < minK or num > maxK:
            bad = i
        if num == minK:
            last_min = i
        if num == maxK:
            last_max = i
        count += max(0, min(last_min, last_max) - bad)
    return count

assert countSubarrays([1,3,5,2,7,5], 1, 5) == 2
assert countSubarrays([1,1,1,1], 1, 1) == 10
assert countSubarrays([1,2,3,4], 2, 3) == 2
print("All tests passed!")

### 19. Remove Covered Intervals (LC 1288)

> 🏢 **Asked by:** Amazon, Google
Remove all intervals that are covered by another interval. Return the number of remaining intervals.

### Approach
Sort by start ascending, then by end descending (so the widest interval with same start comes first). Iterate tracking the max end seen; an interval is covered if its end <= max end.

- **Time**: O(n log n)
- **Space**: O(1)

In [ ]:
def removeCoveredIntervals(intervals):
    intervals.sort(key=lambda x: (x[0], -x[1]))
    remaining = 0
    max_end = 0
    for start, end in intervals:
        if end > max_end:  # not covered
            remaining += 1
            max_end = end
    return remaining

assert removeCoveredIntervals([[1,4],[3,6],[2,8]]) == 2
assert removeCoveredIntervals([[1,4],[2,3]]) == 1
assert removeCoveredIntervals([[1,2],[1,4],[3,4]]) == 1
assert removeCoveredIntervals([[3,10],[4,10],[5,11]]) == 2
print("All tests passed!")

### 20. Count Number of Texts (LC 2266 simplified)

> 🏢 **Asked by:** Amazon, Google
Count the number of possible original messages that could have produced a given key sequence (phone keypad, groups of same digit).

### Approach
DP on groups of consecutive same digits. For a run of k identical digits, the number of ways to decode it uses the same logic as stair climbing (max 3 or 4 steps depending on key).

- **Time**: O(n)
- **Space**: O(n)

In [ ]:
def countTexts(pressedKeys):
    MOD = 10**9 + 7
    n = len(pressedKeys)
    dp = [0] * (n + 1)
    dp[0] = 1
    four_keys = set('79')
    
    for i in range(1, n + 1):
        dp[i] = dp[i-1]  # current key alone
        if i >= 2 and pressedKeys[i-1] == pressedKeys[i-2]:
            dp[i] = (dp[i] + dp[i-2]) % MOD
        if i >= 3 and pressedKeys[i-1] == pressedKeys[i-2] == pressedKeys[i-3]:
            dp[i] = (dp[i] + dp[i-3]) % MOD
        if i >= 4 and pressedKeys[i-1] in four_keys and pressedKeys[i-1] == pressedKeys[i-2] == pressedKeys[i-3] == pressedKeys[i-4]:
            dp[i] = (dp[i] + dp[i-4]) % MOD
    return dp[n]

assert countTexts("22233") == 8
assert countTexts("222222222222222222222222222222222222") == 82876089
assert countTexts("8") == 1
assert countTexts("999") == 4  # 9/99/999/9_9 → wait, 7 ways for 3 nines on key 9
print("All tests passed!")

---
# Medium Problems (15)

### 1. Merge Intervals Full (LC 56)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Given an array of intervals, merge all overlapping intervals and return the result.

### Approach
Sort by start. Walk through and extend or append depending on overlap.

- **Time**: O(n log n)
- **Space**: O(n)

In [ ]:
def mergeFull(intervals):
    intervals.sort(key=lambda x: x[0])
    result = []
    for interval in intervals:
        if result and interval[0] <= result[-1][1]:
            result[-1][1] = max(result[-1][1], interval[1])
        else:
            result.append(interval[:])
    return result

assert mergeFull([[1,3],[2,6],[8,10],[15,18]]) == [[1,6],[8,10],[15,18]]
assert mergeFull([[1,4],[4,5]]) == [[1,5]]
assert mergeFull([[1,4],[0,0]]) == [[0,0],[1,4]]
assert mergeFull([[1,4],[2,3]]) == [[1,4]]
print("All tests passed!")

### 2. Insert Interval (LC 57)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft
Insert a new interval into a sorted non-overlapping list of intervals and merge if necessary.

### Approach
Three phases: (1) add all intervals ending before new start, (2) merge all overlapping with new interval, (3) add all intervals starting after new end.

- **Time**: O(n)
- **Space**: O(n)

In [ ]:
def insert(intervals, newInterval):
    result = []
    i = 0
    n = len(intervals)
    # Phase 1: add intervals ending before new start
    while i < n and intervals[i][1] < newInterval[0]:
        result.append(intervals[i])
        i += 1
    # Phase 2: merge overlapping intervals
    while i < n and intervals[i][0] <= newInterval[1]:
        newInterval[0] = min(newInterval[0], intervals[i][0])
        newInterval[1] = max(newInterval[1], intervals[i][1])
        i += 1
    result.append(newInterval)
    # Phase 3: add remaining
    result.extend(intervals[i:])
    return result

assert insert([[1,3],[6,9]], [2,5]) == [[1,5],[6,9]]
assert insert([[1,2],[3,5],[6,7],[8,10],[12,16]], [4,8]) == [[1,2],[3,10],[12,16]]
assert insert([], [5,7]) == [[5,7]]
assert insert([[1,5]], [2,3]) == [[1,5]]
print("All tests passed!")

### 3. Non-overlapping Intervals (LC 435)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Find the minimum number of intervals to remove to make the rest non-overlapping.

### Approach
Sort by end time. Greedily keep the interval with the earliest end — this maximizes space for future intervals. Count removals.

- **Time**: O(n log n)
- **Space**: O(1)

In [ ]:
def eraseOverlapIntervalsFull(intervals):
    if not intervals:
        return 0
    intervals.sort(key=lambda x: x[1])
    kept = 1
    last_end = intervals[0][1]
    for i in range(1, len(intervals)):
        if intervals[i][0] >= last_end:
            kept += 1
            last_end = intervals[i][1]
    return len(intervals) - kept

assert eraseOverlapIntervalsFull([[1,2],[2,3],[3,4],[1,3]]) == 1
assert eraseOverlapIntervalsFull([[1,2],[1,2],[1,2]]) == 2
assert eraseOverlapIntervalsFull([[1,2],[2,3]]) == 0
assert eraseOverlapIntervalsFull([[-52,31],[-73,-26],[82,97],[-65,-11],[-62,-49],[95,99],[58,95],[-31,49],[66,98],[-63,2],[30,47],[-40,-26]]) == 7
print("All tests passed!")

### 4. Meeting Rooms II (LC 253)

> 🏢 **Asked by:** Amazon, Google, Meta, Microsoft, Bloomberg
Given meeting time intervals, find the minimum number of conference rooms required.

### Approach
Use a min-heap of end times. Sort by start. For each meeting, if it starts after the earliest-ending meeting, reuse that room (pop and push new end). Otherwise add a room.

- **Time**: O(n log n)
- **Space**: O(n)

In [ ]:
import heapq

def minMeetingRooms(intervals):
    if not intervals:
        return 0
    intervals.sort(key=lambda x: x[0])
    heap = []  # min-heap of end times
    for start, end in intervals:
        if heap and heap[0] <= start:
            heapq.heappop(heap)  # reuse room
        heapq.heappush(heap, end)
    return len(heap)

assert minMeetingRooms([[0,30],[5,10],[15,20]]) == 2
assert minMeetingRooms([[7,10],[2,4]]) == 1
assert minMeetingRooms([[1,5],[2,3],[3,7],[4,8]]) == 3
assert minMeetingRooms([]) == 0
print("All tests passed!")

### 5. Minimum Number of Arrows to Burst Balloons (LC 452)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Find the minimum number of arrows to burst all balloons (each arrow travels through all balloons at that x-position).

### Approach
Sort by end. Greedily shoot at the end of the first balloon. Any balloon that starts <= current arrow position is burst. Otherwise, new arrow needed.

- **Time**: O(n log n)
- **Space**: O(1)

In [ ]:
def findMinArrowShots(points):
    points.sort(key=lambda x: x[1])
    arrows = 1
    arrow_pos = points[0][1]
    for start, end in points[1:]:
        if start > arrow_pos:  # current arrow doesn't burst this balloon
            arrows += 1
            arrow_pos = end
    return arrows

assert findMinArrowShots([[10,16],[2,8],[1,6],[7,12]]) == 2
assert findMinArrowShots([[1,2],[3,4],[5,6],[7,8]]) == 4
assert findMinArrowShots([[1,2],[2,3],[3,4],[4,5]]) == 2
assert findMinArrowShots([[1,2]]) == 1
print("All tests passed!")

### 6. Interval List Intersections (LC 986)

> 🏢 **Asked by:** Amazon, Google, Bloomberg
Given two lists of closed intervals, return their intersection.

### Approach
Two-pointer approach. Find the overlap of current pair `[max(lo), min(hi)]`. Advance the pointer with the smaller end.

- **Time**: O(m+n)
- **Space**: O(m+n)

In [ ]:
def intervalIntersection(firstList, secondList):
    i = j = 0
    result = []
    while i < len(firstList) and j < len(secondList):
        lo = max(firstList[i][0], secondList[j][0])
        hi = min(firstList[i][1], secondList[j][1])
        if lo <= hi:
            result.append([lo, hi])
        # advance the one that ends earlier
        if firstList[i][1] < secondList[j][1]:
            i += 1
        else:
            j += 1
    return result

assert intervalIntersection([[0,2],[5,10],[13,23],[24,25]],[[1,5],[8,12],[15,24],[25,26]]) == [[1,2],[5,5],[8,10],[15,23],[24,24],[25,25]]
assert intervalIntersection([[1,3],[5,9]],[]) == []
assert intervalIntersection([],[]) == []
print("All tests passed!")

### 7. My Calendar I (LC 729)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Implement a booking system where a new booking cannot overlap with existing ones.

### Approach
Keep a sorted list of `(start, end)` pairs. For each new booking, binary search for potential conflicts. Two intervals `[s1,e1]` and `[s2,e2]` conflict iff `s1 < e2 and s2 < e1`.

- **Time**: O(n log n) per book
- **Space**: O(n)

In [ ]:
import bisect

class MyCalendar:
    def __init__(self):
        self.bookings = []  # sorted by start
    
    def book(self, start, end):
        # find insertion position by start
        pos = bisect.bisect_right(self.bookings, (start, end))
        # check overlap with previous booking
        if pos > 0 and self.bookings[pos-1][1] > start:
            return False
        # check overlap with next booking
        if pos < len(self.bookings) and end > self.bookings[pos][0]:
            return False
        bisect.insort(self.bookings, (start, end))
        return True

cal = MyCalendar()
assert cal.book(10, 20) == True
assert cal.book(15, 25) == False
assert cal.book(20, 30) == True
assert cal.book(5, 10) == True
assert cal.book(5, 15) == False
print("All tests passed!")

### 8. Divide Intervals Into Minimum Number of Groups (LC 2406)

> 🏢 **Asked by:** Amazon, Google
Divide intervals into groups such that no two intervals in the same group overlap. Return the minimum number of groups.

### Approach
This is identical to Meeting Rooms II. Sort by start. Use a min-heap of group end times. Reuse a group if its end <= current start, else create a new group.

- **Time**: O(n log n)
- **Space**: O(n)

In [ ]:
import heapq

def minGroups(intervals):
    intervals.sort()
    heap = []
    for start, end in intervals:
        if heap and heap[0] < start:
            heapq.heappop(heap)
        heapq.heappush(heap, end)
    return len(heap)

assert minGroups([[5,10],[6,8],[1,5],[2,3],[1,10]]) == 3
assert minGroups([[1,3],[5,6],[8,10],[11,13]]) == 1
assert minGroups([[1,2],[2,3],[3,4]]) == 1  # back-to-back, no true overlap
assert minGroups([[1,4],[2,5],[3,6]]) == 3
print("All tests passed!")

### 9. Car Pooling (LC 1094)

> 🏢 **Asked by:** Amazon, Google, Lyft, Uber
A car with capacity `capacity` makes trips. Each trip picks up and drops off passengers. Can all trips be completed without exceeding capacity?

### Approach
Line sweep / difference array. At each pickup stop, add passengers; at each dropoff stop, remove them. Check if capacity is exceeded at any point.

- **Time**: O(n log n) or O(n + max_stop)
- **Space**: O(max_stop)

In [ ]:
def carPooling(trips, capacity):
    diff = [0] * 1001
    for passengers, start, end in trips:
        diff[start] += passengers
        diff[end] -= passengers
    curr = 0
    for delta in diff:
        curr += delta
        if curr > capacity:
            return False
    return True

assert carPooling([[2,1,5],[3,3,7]], 4) == False
assert carPooling([[2,1,5],[3,3,7]], 5) == True
assert carPooling([[3,2,7],[3,7,9],[8,3,9]], 11) == True
assert carPooling([[2,1,5],[3,5,7]], 3) == True
print("All tests passed!")

### 10. Maximum CPU Load (Classic Interval Scheduling)

> 🏢 **Asked by:** Amazon, Google
Given jobs with `[start, end, load]`, find the maximum CPU load at any time.

### Approach
Sort by start. Use a min-heap of `(end, load)`. Remove jobs that have ended before current start. Track running total load and record the maximum.

- **Time**: O(n log n)
- **Space**: O(n)

In [ ]:
import heapq

def maxCPULoad(jobs):
    jobs.sort(key=lambda x: x[0])
    heap = []  # (end_time, load)
    curr_load = max_load = 0
    for start, end, load in jobs:
        # remove jobs that have ended
        while heap and heap[0][0] <= start:
            curr_load -= heapq.heappop(heap)[1]
        heapq.heappush(heap, (end, load))
        curr_load += load
        max_load = max(max_load, curr_load)
    return max_load

assert maxCPULoad([[1,4,3],[2,5,4],[7,9,6]]) == 7
assert maxCPULoad([[6,7,10],[2,4,11],[8,12,15]]) == 15
assert maxCPULoad([[1,4,2],[2,4,1],[3,6,5]]) == 8
print("All tests passed!")

### 11. Task Scheduler (LC 621)

> 🏢 **Asked by:** Amazon, Google, Microsoft, Meta
Given tasks and a cooldown `n`, find the minimum time units to execute all tasks.

### Approach
The key insight is: the most frequent task determines the minimum frame count. Answer = max(len(tasks), (max_freq-1)*(n+1) + count_of_max_freq_tasks).

- **Time**: O(n)
- **Space**: O(1) (26 letters)

In [ ]:
from collections import Counter

def leastInterval(tasks, n):
    freq = Counter(tasks)
    max_freq = max(freq.values())
    count_max = sum(1 for v in freq.values() if v == max_freq)
    # formula: (max_freq-1) slots of (n+1) plus final group
    return max(len(tasks), (max_freq - 1) * (n + 1) + count_max)

assert leastInterval(["A","A","A","B","B","B"], 2) == 8
assert leastInterval(["A","A","A","B","B","B"], 0) == 6
assert leastInterval(["A","A","A","A","A","A","B","C","D","E","F","G"], 2) == 16
assert leastInterval(["A","B","C","D","E","F"], 2) == 6
print("All tests passed!")

### 12. Remove Covered Intervals (LC 1288) Full

> 🏢 **Asked by:** Amazon, Google
Remove all intervals that are fully covered by another interval in the list.

### Approach
Sort by start ascending, end descending. An interval `[a,b]` is covered by a later one if `b <= max_end_so_far`. Track and count non-covered.

- **Time**: O(n log n)
- **Space**: O(1)

In [ ]:
def removeCoveredFull(intervals):
    intervals.sort(key=lambda x: (x[0], -x[1]))
    count = 0
    max_end = 0
    for _, end in intervals:
        if end > max_end:
            count += 1
            max_end = end
    return count

assert removeCoveredFull([[1,4],[3,6],[2,8]]) == 2
assert removeCoveredFull([[1,4],[2,3]]) == 1
assert removeCoveredFull([[1,2],[1,4],[3,4]]) == 1
assert removeCoveredFull([[1,4],[1,6],[2,8],[5,9]]) == 2
print("All tests passed!")

### 13. Minimum Interval to Include Each Query (LC 2158 simplified)

> 🏢 **Asked by:** Amazon, Google
For each query (integer), find the size of the smallest interval containing it. Return -1 if none.

### Approach
Sort intervals by start; sort queries by value. Use a min-heap (by size). For each query, add all intervals starting <= query; remove expired intervals; answer is smallest remaining.

- **Time**: O((n + q) log n)
- **Space**: O(n + q)

In [ ]:
import heapq

def minInterval(intervals, queries):
    intervals.sort()
    sorted_q = sorted(enumerate(queries), key=lambda x: x[1])
    heap = []  # (size, end)
    ans = [-1] * len(queries)
    i = 0
    for qi, q in sorted_q:
        # add all intervals starting <= q
        while i < len(intervals) and intervals[i][0] <= q:
            s, e = intervals[i]
            heapq.heappush(heap, (e - s + 1, e))
            i += 1
        # remove intervals that end before q
        while heap and heap[0][1] < q:
            heapq.heappop(heap)
        if heap:
            ans[qi] = heap[0][0]
    return ans

assert minInterval([[1,4],[2,4],[3,6],[4,4]], [2,3,4,5]) == [3,3,1,4]
assert minInterval([[2,3],[2,5],[1,8],[20,25]], [2,19,5,22]) == [2,-1,4,6]
print("All tests passed!")

### 14. Average Waiting Time (LC 1701)

> 🏢 **Asked by:** Amazon, Google
Customers arrive at a restaurant one at a time. Given `customers[i] = [arrival, time]`, return the average waiting time.

### Approach
Simulate: track when the chef becomes free (`free_at`). For each customer, service starts at `max(free_at, arrival)`. Waiting time = finish - arrival.

- **Time**: O(n)
- **Space**: O(1)

In [ ]:
def averageWaitingTime(customers):
    total_wait = 0
    free_at = 0
    for arrival, cook_time in customers:
        free_at = max(free_at, arrival) + cook_time
        total_wait += free_at - arrival
    return total_wait / len(customers)

assert abs(averageWaitingTime([[1,2],[2,5],[4,3]]) - 5.0) < 1e-9
assert abs(averageWaitingTime([[5,2],[5,4],[10,3],[20,1]]) - 3.25) < 1e-9
print("All tests passed!")

### 15. My Calendar I Full (LC 729) — SortedList variant

> 🏢 **Asked by:** Amazon, Google, Microsoft
Full implementation with O(log n) book using sorted list and binary search.

### Approach
Maintain a sorted list of bookings. For each new `[start, end)`, use binary search to find if any existing booking overlaps. Overlap: the predecessor must end <= start and successor must start >= end.

- **Time**: O(n) insert, O(log n) search
- **Space**: O(n)

In [ ]:
import bisect

class MyCalendarFull:
    def __init__(self):
        self.starts = []
        self.ends = []
    
    def book(self, start, end):
        # find where to insert
        pos = bisect.bisect_right(self.starts, start)
        # check overlap with previous interval
        if pos > 0 and self.ends[pos - 1] > start:
            return False
        # check overlap with next interval
        if pos < len(self.starts) and self.starts[pos] < end:
            return False
        self.starts.insert(pos, start)
        self.ends.insert(pos, end)
        return True

mc = MyCalendarFull()
assert mc.book(10, 20) == True
assert mc.book(15, 25) == False
assert mc.book(20, 30) == True
assert mc.book(5, 10) == True
assert mc.book(25, 35) == True
assert mc.book(5, 11) == False
print("All tests passed!")

---
# Hard Problems (10)

### 1. The Skyline Problem (LC 218)

> 🏢 **Asked by:** Google, Amazon, Microsoft, Bloomberg
Given `buildings[i] = [left, right, height]`, return the skyline as a list of key points.

### Approach
Events-based approach with a max-heap. Create events: building start (negative height to enter heap) and end (height 0 to trigger removal). Sort events. Use a multiset/counter heap to track active heights.

- **Time**: O(n log n)
- **Space**: O(n)

In [ ]:
import heapq
from collections import defaultdict

def getSkyline(buildings):
    events = []
    for l, r, h in buildings:
        events.append((l, -h))  # start: negative height
        events.append((r, 0))   # end: height 0
    events.sort()
    
    result = []
    heap = [0]  # max-heap (use negative values)
    active = defaultdict(int)
    active[0] = 1
    prev_max = 0
    
    for x, h in events:
        if h != 0:  # start event
            heapq.heappush(heap, h)  # h is already negative
            active[-h] += 1
        else:  # end event
            # find what building ended here — mark its height for removal
            # we need to find which building this end belongs to
            # actually we need to process differently: match start/end
            pass
    
    # Cleaner approach using sorted events
    events2 = []
    for l, r, h in buildings:
        events2.append((l, -h, r))  # start with neg height and end
        events2.append((r, 0, 0))   # end
    events2.sort()
    
    heap2 = [(0, float('inf'))]  # (neg_height, end)
    result2 = []
    
    for x, neg_h, end in events2:
        if neg_h != 0:  # building start
            heapq.heappush(heap2, (neg_h, end))
        # clean up ended buildings
        while heap2[0][1] <= x:
            heapq.heappop(heap2)
        curr_max = -heap2[0][0]
        if curr_max != prev_max:
            result2.append([x, curr_max])
            prev_max = curr_max
    return result2

assert getSkyline([[2,9,10],[3,7,15],[5,12,12],[15,20,10],[19,24,8]]) == [[2,10],[3,15],[7,12],[12,0],[15,10],[20,8],[24,0]]
assert getSkyline([[0,2,3],[2,5,3]]) == [[0,3],[5,0]]
print("All tests passed!")

### 2. Range Module (LC 715)

> 🏢 **Asked by:** Google, Amazon
Implement a range module supporting addRange, queryRange, and removeRange operations.

### Approach
Maintain a sorted list of disjoint intervals. For add/remove, find overlapping intervals using binary search and merge/split accordingly.

- **Time**: O(n) per operation worst case, O(log n) for search
- **Space**: O(n)

In [ ]:
import bisect

class RangeModule:
    def __init__(self):
        self.ranges = []  # sorted list of [start, end]
    
    def addRange(self, left, right):
        new = [left, right]
        result = []
        inserted = False
        for r in self.ranges:
            if r[1] < new[0]:  # r ends before new starts
                result.append(r)
            elif r[0] > new[1]:  # r starts after new ends
                if not inserted:
                    result.append(new)
                    inserted = True
                result.append(r)
            else:  # overlap — merge
                new[0] = min(new[0], r[0])
                new[1] = max(new[1], r[1])
        if not inserted:
            result.append(new)
        self.ranges = result
    
    def queryRange(self, left, right):
        for r in self.ranges:
            if r[0] <= left and right <= r[1]:
                return True
        return False
    
    def removeRange(self, left, right):
        result = []
        for r in self.ranges:
            if r[1] <= left or r[0] >= right:
                result.append(r)
            else:
                if r[0] < left:
                    result.append([r[0], left])
                if r[1] > right:
                    result.append([right, r[1]])
        self.ranges = result

rm = RangeModule()
rm.addRange(10, 20)
rm.removeRange(14, 16)
assert rm.queryRange(10, 14) == True
assert rm.queryRange(13, 15) == False
assert rm.queryRange(16, 17) == True
rm.addRange(13, 15)
assert rm.queryRange(13, 15) == True
print("All tests passed!")

### 3. My Calendar III (LC 732)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Return the maximum k-booking (maximum number of overlapping events at any point).

### Approach
Difference array using a sorted dict (TreeMap). Increment at start, decrement at end. Compute prefix sums to find max overlap after each booking.

- **Time**: O(n log n) per book
- **Space**: O(n)

In [ ]:
from sortedcontainers import SortedDict

class MyCalendarThree:
    def __init__(self):
        self.timeline = SortedDict()
    
    def book(self, start, end):
        self.timeline[start] = self.timeline.get(start, 0) + 1
        self.timeline[end] = self.timeline.get(end, 0) - 1
        max_k = curr = 0
        for delta in self.timeline.values():
            curr += delta
            max_k = max(max_k, curr)
        return max_k

mc3 = MyCalendarThree()
assert mc3.book(10, 20) == 1
assert mc3.book(50, 60) == 1
assert mc3.book(10, 40) == 2
assert mc3.book(5, 15) == 3
assert mc3.book(5, 10) == 3
assert mc3.book(25, 55) == 3
print("All tests passed!")

### 4. Data Stream as Disjoint Intervals (LC 352)

> 🏢 **Asked by:** Google, Amazon
Given a stream of integers, maintain and return all disjoint intervals they form.

### Approach
Use a sorted list of intervals. On each `addNum`, find neighbors by binary search and merge if adjacent or overlapping.

- **Time**: O(n log n) total
- **Space**: O(n)

In [ ]:
import bisect

class SummaryRanges:
    def __init__(self):
        self.intervals = []  # sorted by start
    
    def addNum(self, val):
        new = [val, val]
        result = []
        inserted = False
        for iv in self.intervals:
            if iv[1] + 1 < new[0]:  # iv ends before new (with gap)
                result.append(iv)
            elif iv[0] - 1 > new[1]:  # iv starts after new (with gap)
                if not inserted:
                    result.append(new)
                    inserted = True
                result.append(iv)
            else:  # adjacent or overlapping — merge
                new[0] = min(new[0], iv[0])
                new[1] = max(new[1], iv[1])
        if not inserted:
            result.append(new)
        self.intervals = result
    
    def getIntervals(self):
        return self.intervals

sr = SummaryRanges()
sr.addNum(1)
assert sr.getIntervals() == [[1,1]]
sr.addNum(3)
assert sr.getIntervals() == [[1,1],[3,3]]
sr.addNum(7)
sr.addNum(2)
assert sr.getIntervals() == [[1,3],[7,7]]
sr.addNum(6)
assert sr.getIntervals() == [[1,3],[6,7]]
print("All tests passed!")

### 5. Minimum Number of Taps to Open to Water a Garden (LC 1326)

> 🏢 **Asked by:** Amazon, Google
A garden of length `n` has taps at positions 0..n. Each tap `i` waters `[i-ranges[i], i+ranges[i]]`. Find minimum taps to water the whole garden.

### Approach
Convert to an interval coverage problem. Sort intervals by start. Use a greedy "jump game" approach: from each position, find the interval with the farthest reach.

- **Time**: O(n log n)
- **Space**: O(n)

In [ ]:
def minTaps(n, ranges):
    # Convert taps to intervals
    intervals = []
    for i, r in enumerate(ranges):
        intervals.append([max(0, i - r), min(n, i + r)])
    intervals.sort()
    
    # Greedy interval cover
    taps = 0
    curr_end = 0  # current coverage reach
    max_end = 0   # farthest reachable by any interval starting <= curr_end
    i = 0
    while curr_end < n:
        while i < len(intervals) and intervals[i][0] <= curr_end:
            max_end = max(max_end, intervals[i][1])
            i += 1
        if max_end == curr_end:  # no progress — impossible
            return -1
        taps += 1
        curr_end = max_end
    return taps

assert minTaps(5, [3,4,1,1,0,0]) == 1
assert minTaps(3, [0,0,0,0]) == -1
assert minTaps(7, [1,2,1,0,2,1,0,1]) == 3
assert minTaps(8, [4,0,0,0,4,0,0,0,4]) == 2
print("All tests passed!")

### 6. Maximum Number of Events That Can Be Attended II (LC 1751)

> 🏢 **Asked by:** Amazon, Google
Attend at most `k` non-overlapping events to maximize total value.

### Approach
Sort by end. DP: `dp[i][j]` = max value attending j events from first i events. For each event i, binary search for the last event ending before event i starts.

- **Time**: O(n log n + nk)
- **Space**: O(nk)

In [ ]:
import bisect

def maxValue(events, k):
    events.sort(key=lambda x: x[1])
    n = len(events)
    ends = [e[1] for e in events]
    # dp[i][j] = max value using first i events, attending exactly <=j
    dp = [[0] * (k + 1) for _ in range(n + 1)]
    
    for i in range(1, n + 1):
        s, e, v = events[i-1]
        # find last event that ends before s
        prev = bisect.bisect_left(ends, s, 0, i-1)
        for j in range(1, k + 1):
            dp[i][j] = max(dp[i-1][j], dp[prev][j-1] + v)
    return dp[n][k]

assert maxValue([[1,2,4],[3,4,3],[2,3,1]], 2) == 7
assert maxValue([[1,2,4],[3,4,3],[2,3,10]], 2) == 10
assert maxValue([[1,1,1],[2,2,2],[3,3,3],[4,4,4]], 3) == 9
assert maxValue([[1,2,4],[3,4,3],[2,3,1]], 3) == 8
print("All tests passed!")

### 7. Falling Squares (LC 699)

> 🏢 **Asked by:** Google, Amazon
Squares fall on an infinite number line. Return the height of the tallest stack after each square falls.

### Approach
For each new square, find the maximum height of all previously landed squares that overlap with it. The new square lands on top: its height = that max + its own side length.

- **Time**: O(n²) naive
- **Space**: O(n)

In [ ]:
def fallingSquares(positions):
    # intervals[i] = (left, right, height) of landed squares
    landed = []  # (left, right, height)
    result = []
    global_max = 0
    
    for left, size in positions:
        right = left + size
        # find max height of overlapping landed squares
        base = 0
        for l, r, h in landed:
            if l < right and r > left:  # overlap
                base = max(base, h)
        new_height = base + size
        landed.append((left, right, new_height))
        global_max = max(global_max, new_height)
        result.append(global_max)
    
    return result

assert fallingSquares([[1,2],[2,2],[1,2]]) == [2,4,5]
assert fallingSquares([[100,100],[200,100]]) == [100,100]
assert fallingSquares([[1,2],[1,2],[1,2]]) == [2,4,6]
print("All tests passed!")

### 8. Employee Free Time (LC 759)

> 🏢 **Asked by:** Amazon, Google
Given a list of employees' schedules, find the free time common to all employees.

### Approach
Collect all intervals, sort, merge. Then find the gaps between merged intervals — those are the free times.

- **Time**: O(n log n)
- **Space**: O(n)

In [ ]:
def employeeFreeTime(schedule):
    # schedule is a list of lists of [start, end] intervals per employee
    all_intervals = []
    for employee in schedule:
        for interval in employee:
            all_intervals.append(interval)
    all_intervals.sort()
    
    # Merge all intervals
    merged = [all_intervals[0][:]]
    for s, e in all_intervals[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    
    # Free time = gaps between merged intervals
    free = []
    for i in range(1, len(merged)):
        free.append([merged[i-1][1], merged[i][0]])
    return free

assert employeeFreeTime([[[1,3],[6,7]],[[2,4]],[[2,5],[9,12]]]) == [[5,6],[7,9]]
assert employeeFreeTime([[[1,3],[9,12]],[[2,4]],[[6,8]]]) == [[4,6],[8,9]]
assert employeeFreeTime([[[1,2],[5,10]],[[3,6]]]) == [[2,3]]
print("All tests passed!")

### 9. Russian Doll Envelopes (LC 354) — Interval Nesting

> 🏢 **Asked by:** Google, Amazon, Bloomberg
Find the maximum number of envelopes that can be nested (both width and height strictly increasing).

### Approach
Sort by width ascending, height descending (to prevent multiple same-width envelopes). Then find LIS on heights. The descending height sort ensures we pick at most one per width.

- **Time**: O(n log n)
- **Space**: O(n)

In [ ]:
import bisect

def maxEnvelopesInterval(envelopes):
    envelopes.sort(key=lambda x: (x[0], -x[1]))
    tails = []
    for _, h in envelopes:
        pos = bisect.bisect_left(tails, h)
        if pos == len(tails):
            tails.append(h)
        else:
            tails[pos] = h
    return len(tails)

assert maxEnvelopesInterval([[5,4],[6,4],[6,7],[2,3]]) == 3
assert maxEnvelopesInterval([[1,1],[1,1],[1,1]]) == 1
assert maxEnvelopesInterval([[1,3],[3,5],[6,7],[6,8],[8,4],[9,5]]) == 4
print("All tests passed!")

### 10. Minimum Cost to Cut a Stick (LC 1547)

> 🏢 **Asked by:** Amazon, Google
Given a stick of length `n` and positions to cut, return the minimum total cost to make all cuts (cost = length of stick being cut).

### Approach
Interval DP. Add 0 and n as boundaries. Sort cuts. `dp[i][j]` = min cost to cut the segment `[cuts[i], cuts[j]]`. For each segment, try all cuts between i and j.

- **Time**: O(m³) where m = len(cuts)
- **Space**: O(m²)

In [ ]:
def minCost(n, cuts):
    cuts = sorted([0] + cuts + [n])
    m = len(cuts)
    dp = [[0] * m for _ in range(m)]
    
    # length = number of intervals between i and j
    for length in range(2, m):
        for i in range(m - length):
            j = i + length
            dp[i][j] = float('inf')
            for k in range(i + 1, j):
                cost = cuts[j] - cuts[i] + dp[i][k] + dp[k][j]
                dp[i][j] = min(dp[i][j], cost)
    
    return dp[0][m-1]

assert minCost(7, [1,3,4,5]) == 16
assert minCost(9, [5,6,1,4,2]) == 22
assert minCost(10, [2,4,7]) == 20
print("All tests passed!")

---
## Easy Problems (21–40)

### Easy 21 — Merge Two 2D Arrays by Summing Values (LC 2570)

> 🏢 **Asked by:** Amazon, Google
Given two 2-D arrays `nums1` and `nums2` where each row is `[id, val]`, merge them by summing values with the same id and return sorted by id.

### Approach
Use a dictionary keyed by id to accumulate sums, then sort by key.

**Time:** O((m+n) log(m+n)) | **Space:** O(m+n)

In [ ]:
def mergeArrays(nums1, nums2):
    from collections import defaultdict
    totals = defaultdict(int)
    for id_, val in nums1:
        totals[id_] += val
    for id_, val in nums2:
        totals[id_] += val
    return sorted([k, v] for k, v in totals.items())

assert mergeArrays([[1,2],[2,3],[4,5]],[[1,4],[3,2],[4,1]]) == [[1,6],[2,3],[3,2],[4,6]]
assert mergeArrays([[2,4],[3,6],[5,5]],[[1,3],[4,3]]) == [[1,3],[2,4],[3,6],[4,3],[5,5]]
assert mergeArrays([],[]) == []
print("All tests passed!")

### Easy 22 — Count Days Without Meetings (LC 3169)

> 🏢 **Asked by:** Amazon, Google
Given `days` total workdays and `meetings` list of `[start, end]` intervals, return the number of days with no meeting (intervals may overlap).

### Approach
Merge the meeting intervals, then subtract total covered days from `days`.

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def countDays(days, meetings):
    meetings.sort()
    merged = []
    for s, e in meetings:
        if merged and s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    busy = sum(e - s + 1 for s, e in merged)
    return days - busy

assert countDays(10, [[5,7],[1,3],[9,10]]) == 2
assert countDays(5, [[2,4],[1,3]]) == 1
assert countDays(6, [[1,6]]) == 0
print("All tests passed!")

### Easy 23 — Convert Character Ranges to Interval List

> 🏢 **Asked by:** Amazon, Google
Given a string of distinct lowercase letters, return sorted intervals `[start_ord, end_ord]` of consecutive ASCII character runs.

### Approach
Sort the characters, then scan for consecutive runs using the same summary-ranges technique applied to ordinal values.

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def charRanges(s):
    nums = sorted(ord(c) for c in s)
    if not nums:
        return []
    result = []
    start = nums[0]
    for i in range(1, len(nums)):
        if nums[i] != nums[i-1] + 1:
            result.append([start, nums[i-1]])
            start = nums[i]
    result.append([start, nums[-1]])
    return result

assert charRanges('abcef') == [[97,99],[101,102]]
assert charRanges('z') == [[122,122]]
assert charRanges('') == []
assert charRanges('ace') == [[97,97],[99,99],[101,101]]
print("All tests passed!")

### Easy 24 — Find the Intersection of Two Lists as Ranges

> 🏢 **Asked by:** Amazon, Google
Given two sorted lists of integers, return the sorted list of ranges that appear in both lists.

### Approach
Convert each list to a set, find the intersection, sort it, then apply summary-ranges to compress consecutive values.

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def intersectAsRanges(a, b):
    common = sorted(set(a) & set(b))
    if not common:
        return []
    result = []
    start = common[0]
    for i in range(1, len(common)):
        if common[i] != common[i-1] + 1:
            result.append([start, common[i-1]])
            start = common[i]
    result.append([start, common[-1]])
    return result

assert intersectAsRanges([1,2,3,5,6],[2,3,4,6,7]) == [[2,3],[6,6]]
assert intersectAsRanges([1,2],[3,4]) == []
assert intersectAsRanges([1,2,3],[1,2,3]) == [[1,3]]
print("All tests passed!")

### Easy 25 — Flowers in Bloom at Time t (LC 2251 simplified)

> 🏢 **Asked by:** Amazon, Google
Given `flowers` list of `[start, end]` bloom periods and a single query time `t`, return how many flowers are blooming at time `t`.

### Approach
Iterate through each flower's interval and count those where `start <= t <= end`.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def flowersAtTime(flowers, t):
    return sum(1 for s, e in flowers if s <= t <= e)

assert flowersAtTime([[1,6],[3,7],[9,12],[4,13]], 4) == 3
assert flowersAtTime([[1,6],[3,7],[9,12],[4,13]], 9) == 2
assert flowersAtTime([[1,2],[3,4]], 5) == 0
assert flowersAtTime([], 1) == 0
print("All tests passed!")

### Easy 26 — Rectangle Overlap (LC 836)

> 🏢 **Asked by:** Amazon, Google
Two axis-aligned rectangles are given as `[x1,y1,x2,y2]`. Return `True` if they overlap (sharing a border does not count).

### Approach
Two rectangles do NOT overlap iff one is entirely to the left/right/above/below the other. Negate that condition.

**Time:** O(1) | **Space:** O(1)

In [ ]:
def isRectangleOverlap(rec1, rec2):
    return not (rec1[2] <= rec2[0] or rec2[2] <= rec1[0] or
                rec1[3] <= rec2[1] or rec2[3] <= rec1[1])

assert isRectangleOverlap([0,0,2,2],[1,1,3,3]) == True
assert isRectangleOverlap([0,0,1,1],[1,0,2,1]) == False
assert isRectangleOverlap([0,0,2,2],[3,3,5,5]) == False
assert isRectangleOverlap([0,0,3,3],[1,1,2,2]) == True
print("All tests passed!")

### Easy 27 — Largest Number At Least Twice of Others (LC 747)

> 🏢 **Asked by:** Amazon, Google
Find the index of the largest element in `nums` if it is at least twice as large as every other element. Return -1 otherwise.

### Approach
Find the largest and second-largest values. If largest >= 2 * second_largest, return its index.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def dominantIndex(nums):
    if len(nums) == 1:
        return 0
    idx = nums.index(max(nums))
    m = nums[idx]
    for i, v in enumerate(nums):
        if i != idx and m < 2 * v:
            return -1
    return idx

assert dominantIndex([3,6,1,0]) == 1
assert dominantIndex([1,2,3,4]) == -1
assert dominantIndex([1]) == 0
assert dominantIndex([0,0,0,1]) == 3
print("All tests passed!")

### Easy 28 — Maximum Difference Between Adjacent Elements

> 🏢 **Asked by:** Amazon, Google
Given an array of integers, return the maximum absolute difference between any two adjacent elements.

### Approach
Iterate through consecutive pairs and track the running maximum of `abs(nums[i] - nums[i-1])`.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def maxAdjacentDiff(nums):
    return max(abs(nums[i] - nums[i-1]) for i in range(1, len(nums)))

assert maxAdjacentDiff([1, 5, 2, 10]) == 8
assert maxAdjacentDiff([1, 2, 3, 4]) == 1
assert maxAdjacentDiff([10, 1]) == 9
assert maxAdjacentDiff([3, 3, 3]) == 0
print("All tests passed!")

### Easy 29 — Employee Shift Interval Coverage

> 🏢 **Asked by:** Amazon, Google
Given a list of employee `[start, end]` shift intervals, return the total number of hours covered (merged, no double-counting).

### Approach
Sort and merge overlapping intervals, then sum `end - start` for each merged interval.

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def totalShiftCoverage(shifts):
    if not shifts:
        return 0
    shifts.sort()
    merged = [list(shifts[0])]
    for s, e in shifts[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return sum(e - s for s, e in merged)

assert totalShiftCoverage([[1,4],[2,6],[8,10]]) == 7
assert totalShiftCoverage([[1,3],[3,5]]) == 4
assert totalShiftCoverage([]) == 0
assert totalShiftCoverage([[0,24]]) == 24
print("All tests passed!")

### Easy 30 — Minimum Taps to Water Garden (Small Input)

> 🏢 **Asked by:** Amazon, Google
Garden of length `n`. Each tap `i` waters `[i - ranges[i], i + ranges[i]]`. Return minimum taps to water `[0, n]`, or -1 if impossible.

### Approach
Convert each tap to an interval, then apply the greedy interval cover algorithm: at each position, extend as far right as possible.

**Time:** O(n) | **Space:** O(n)

In [ ]:
def minTaps(n, ranges):
    max_reach = [0] * (n + 1)
    for i, r in enumerate(ranges):
        left = max(0, i - r)
        max_reach[left] = max(max_reach[left], i + r)
    taps = cur_end = far = 0
    for i in range(n + 1):
        if i > cur_end:
            return -1
        if i > far:
            taps += 1
            far = cur_end
        cur_end = max(cur_end, max_reach[i])
    return taps

assert minTaps(5, [3,4,1,1,0,0]) == 1
assert minTaps(3, [0,0,0,0]) == -1
assert minTaps(7, [1,2,1,0,2,1,0,1]) == 3
assert minTaps(0, [0]) == 0
print("All tests passed!")

### Easy 31 — Partition Labels (LC 763)

> 🏢 **Asked by:** Amazon, Google
Partition string `s` into as many parts as possible so each letter appears in at most one part. Return sizes of the parts.

### Approach
Record the last occurrence of each character. Use a sweep: extend the current partition's end whenever we see a character whose last index is beyond the current end. Each time `i == end`, we close a partition.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def partitionLabels(s):
    last = {c: i for i, c in enumerate(s)}
    result = []
    start = end = 0
    for i, c in enumerate(s):
        end = max(end, last[c])
        if i == end:
            result.append(end - start + 1)
            start = i + 1
    return result

assert partitionLabels('ababcbacadefegdehijhklij') == [9,7,8]
assert partitionLabels('eccbbbbdec') == [10]
assert partitionLabels('a') == [1]
assert partitionLabels('ab') == [1,1]
print("All tests passed!")

### Easy 32 — Find All Intervals That Overlap With a Given Interval

> 🏢 **Asked by:** Amazon, Google
Given a list of intervals and a query interval `q`, return all intervals from the list that overlap with `q`.

### Approach
Two intervals `[a,b]` and `[c,d]` overlap iff `a <= d` and `c <= b`. Filter the list with this condition.

**Time:** O(n) | **Space:** O(k) where k = number of results

In [ ]:
def findOverlapping(intervals, q):
    qs, qe = q
    return [iv for iv in intervals if iv[0] <= qe and qs <= iv[1]]

ivs = [[1,3],[2,6],[8,10],[15,18]]
assert findOverlapping(ivs, [2,9]) == [[1,3],[2,6],[8,10]]
assert findOverlapping(ivs, [4,7]) == [[2,6]]
assert findOverlapping(ivs, [11,14]) == []
assert findOverlapping(ivs, [1,18]) == [[1,3],[2,6],[8,10],[15,18]]
print("All tests passed!")

### Easy 33 — Count Overlapping Pairs of Intervals (Naïve)

> 🏢 **Asked by:** Amazon, Google
Given a list of intervals, count the number of pairs `(i, j)` with `i < j` where the intervals overlap.

### Approach
Check every pair with the standard overlap condition `a[0] <= b[1] and b[0] <= a[1]`.

**Time:** O(n²) | **Space:** O(1)

In [ ]:
def countOverlappingPairs(intervals):
    count = 0
    n = len(intervals)
    for i in range(n):
        for j in range(i+1, n):
            a, b = intervals[i], intervals[j]
            if a[0] <= b[1] and b[0] <= a[1]:
                count += 1
    return count

assert countOverlappingPairs([[1,3],[2,4],[5,7]]) == 1
assert countOverlappingPairs([[1,5],[2,3],[4,6]]) == 2
assert countOverlappingPairs([[1,2],[3,4]]) == 0
assert countOverlappingPairs([[1,10],[2,9],[3,8]]) == 3
print("All tests passed!")

### Easy 34 — Merge Overlapping Intervals (Return Count Merged)

> 🏢 **Asked by:** Amazon, Google
Given a list of intervals, merge all overlapping ones and return the number of merged intervals in the result.

### Approach
Sort by start, perform standard merge, return `len(merged)`.

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def countAfterMerge(intervals):
    if not intervals:
        return 0
    intervals.sort()
    merged = [list(intervals[0])]
    for s, e in intervals[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return len(merged)

assert countAfterMerge([[1,3],[2,6],[8,10],[15,18]]) == 3
assert countAfterMerge([[1,4],[4,5]]) == 1
assert countAfterMerge([[1,2],[3,4],[5,6]]) == 3
assert countAfterMerge([]) == 0
print("All tests passed!")

### Easy 35 — Find Gaps Between Intervals

> 🏢 **Asked by:** Amazon, Google
Given a sorted list of non-overlapping intervals within a universe `[lo, hi]`, return the list of gap intervals not covered.

### Approach
Walk through the sorted intervals. A gap exists between the end of the previous interval and the start of the next one.

**Time:** O(n) | **Space:** O(n)

In [ ]:
def findGaps(intervals, lo, hi):
    gaps = []
    prev = lo
    for s, e in sorted(intervals):
        if prev < s:
            gaps.append([prev, s])
        prev = max(prev, e)
    if prev < hi:
        gaps.append([prev, hi])
    return gaps

assert findGaps([[2,5],[7,9]], 0, 10) == [[0,2],[5,7],[9,10]]
assert findGaps([[0,10]], 0, 10) == []
assert findGaps([], 1, 5) == [[1,5]]
assert findGaps([[1,2],[3,4]], 0, 5) == [[0,1],[2,3],[4,5]]
print("All tests passed!")

### Easy 36 — Find Maximum Overlap at Any Point

> 🏢 **Asked by:** Amazon, Google
Given a list of intervals `[start, end]`, return the maximum number of intervals that overlap at any single point.

### Approach
Sweep-line: push +1 events for start and -1 events for end+1, sort, and track the running sum maximum.

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def maxOverlap(intervals):
    events = []
    for s, e in intervals:
        events.append((s, 1))
        events.append((e + 1, -1))
    events.sort()
    cur = res = 0
    for _, delta in events:
        cur += delta
        res = max(res, cur)
    return res

assert maxOverlap([[1,3],[2,4],[3,5]]) == 3
assert maxOverlap([[1,2],[3,4],[5,6]]) == 1
assert maxOverlap([[1,10],[2,9],[3,8],[4,7]]) == 4
assert maxOverlap([[1,1]]) == 1
print("All tests passed!")

### Easy 37 — Sort and Merge Bookings

> 🏢 **Asked by:** Amazon, Google
Given a list of bookings `[start, end, price]`, merge overlapping bookings (by time) and return merged `[start, end]` segments sorted.

### Approach
Extract `[start, end]` pairs, sort by start, then apply standard interval merge.

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def mergeBookings(bookings):
    intervals = sorted([s, e] for s, e, _ in bookings)
    if not intervals:
        return []
    merged = [intervals[0]]
    for s, e in intervals[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return merged

assert mergeBookings([[1,3,50],[2,5,80],[7,9,30]]) == [[1,5],[7,9]]
assert mergeBookings([[1,2,10],[3,4,20]]) == [[1,2],[3,4]]
assert mergeBookings([]) == []
print("All tests passed!")

### Easy 38 — Check if Schedule is Conflict-Free

> 🏢 **Asked by:** Amazon, Google
Given a list of `[start, end]` time slots, return `True` if no two slots overlap (touching at an endpoint is NOT a conflict).

### Approach
Sort by start. Check each consecutive pair: if previous end > current start, a conflict exists.

**Time:** O(n log n) | **Space:** O(1)

In [ ]:
def isConflictFree(slots):
    slots.sort()
    for i in range(1, len(slots)):
        if slots[i][0] < slots[i-1][1]:
            return False
    return True

assert isConflictFree([[1,2],[3,4],[5,6]]) == True
assert isConflictFree([[1,3],[2,4]]) == False
assert isConflictFree([[1,2],[2,3]]) == True
assert isConflictFree([]) == True
print("All tests passed!")

### Easy 39 — Longest Non-Overlapping Intervals (Greedy)

> 🏢 **Asked by:** Amazon, Google
Given a list of `[start, end]` intervals, return the maximum count of non-overlapping intervals you can select.

### Approach
Greedy: sort by end time, always pick the interval with the earliest finish that starts after the last chosen interval ends.

**Time:** O(n log n) | **Space:** O(1)

In [ ]:
def maxNonOverlapping(intervals):
    intervals.sort(key=lambda x: x[1])
    count = 0
    end = float('-inf')
    for s, e in intervals:
        if s >= end:
            count += 1
            end = e
    return count

assert maxNonOverlapping([[1,2],[2,3],[3,4],[1,3]]) == 3
assert maxNonOverlapping([[1,2],[1,2],[1,2]]) == 1
assert maxNonOverlapping([[1,2],[2,3]]) == 2
assert maxNonOverlapping([]) == 0
print("All tests passed!")

### Easy 40 — Find Union Length of All Intervals (LC 3224)

> 🏢 **Asked by:** Amazon, Google
Given a list of `[start, end]` intervals (inclusive), return the total length covered by their union.

### Approach
Sort and merge intervals, then sum `end - start + 1` for each merged interval (treating endpoints as integers).

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def unionLength(intervals):
    if not intervals:
        return 0
    intervals = sorted(intervals)
    merged = [list(intervals[0])]
    for s, e in intervals[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return sum(e - s + 1 for s, e in merged)

assert unionLength([[1,3],[2,5],[7,9]]) == 8
assert unionLength([[1,4],[4,8]]) == 8
assert unionLength([[1,1],[2,2],[3,3]]) == 3
assert unionLength([]) == 0
print("All tests passed!")

---
## Medium Problems (16–30)

### Medium 16 — My Calendar II (LC 731)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Implement a calendar that allows booking `[start, end)`. A booking is allowed unless it causes a triple overlap. Return `True` if the booking succeeds.

### Approach
Maintain two lists: `single` (booked once) and `double` (booked twice). For each new booking, check if it intersects anything in `double`. If not, add intersections with `single` to `double`, then add the new booking to `single`.

**Time:** O(n²) | **Space:** O(n)

In [ ]:
class MyCalendarTwo:
    def __init__(self):
        self.single = []
        self.double = []

    def _overlap(self, a, b, c, d):
        return max(a, c) < min(b, d)

    def book(self, start, end):
        for s, e in self.double:
            if self._overlap(start, end, s, e):
                return False
        for s, e in self.single:
            if self._overlap(start, end, s, e):
                self.double.append([max(start, s), min(end, e)])
        self.single.append([start, end])
        return True

cal = MyCalendarTwo()
assert cal.book(10, 20) == True
assert cal.book(50, 60) == True
assert cal.book(10, 40) == True
assert cal.book(5, 15) == False
assert cal.book(5, 10) == True
assert cal.book(25, 55) == True
print("All tests passed!")

### Medium 17 — Maximum Number of Events That Can Be Attended (LC 1353)

> 🏢 **Asked by:** Amazon, Google
Given `events` where `events[i] = [startDay, endDay]`, you can attend one event per day. Return the maximum number of events you can attend.

### Approach
Sort by start day. Use a min-heap of end days for events available on each day. Greedily attend the earliest-ending available event.

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
import heapq

def maxEvents(events):
    events.sort()
    heap = []
    i = 0
    n = len(events)
    count = 0
    day = 1
    max_day = max(e for _, e in events)
    while i < n or heap:
        while i < n and events[i][0] == day:
            heapq.heappush(heap, events[i][1])
            i += 1
        while heap and heap[0] < day:
            heapq.heappop(heap)
        if heap:
            heapq.heappop(heap)
            count += 1
        day += 1
        if day > max_day:
            break
    return count

assert maxEvents([[1,2],[2,3],[3,4]]) == 3
assert maxEvents([[1,2],[2,3],[3,4],[1,2]]) == 4
assert maxEvents([[1,4],[4,4],[2,2],[3,4],[1,1]]) == 4
print("All tests passed!")

### Medium 18 — Minimum Arrows from the Top (Vertical Arrow Variant)

> 🏢 **Asked by:** Amazon, Google
Balloons float horizontally. Each balloon spans `[x_start, x_end]` on the x-axis. An arrow shot at x bursts every balloon containing x. Return minimum arrows needed to burst all balloons.

### Approach
Sort by end. Greedily shoot at the end of the first unbursted balloon; this also bursts all overlapping balloons whose start <= that end.

**Time:** O(n log n) | **Space:** O(1)

In [ ]:
def findMinArrowShots(points):
    if not points:
        return 0
    points.sort(key=lambda x: x[1])
    arrows = 1
    arrow_pos = points[0][1]
    for start, end in points[1:]:
        if start > arrow_pos:
            arrows += 1
            arrow_pos = end
    return arrows

assert findMinArrowShots([[10,16],[2,8],[1,6],[7,12]]) == 2
assert findMinArrowShots([[1,2],[3,4],[5,6],[7,8]]) == 4
assert findMinArrowShots([[1,2],[2,3],[3,4],[4,5]]) == 2
assert findMinArrowShots([]) == 0
print("All tests passed!")

### Medium 19 — Interval Schedule Maximization (Weighted Simplified)

> 🏢 **Asked by:** Amazon, Google
Given jobs `[start, end, profit]`, select non-overlapping jobs to maximize total profit (simplified: jobs sorted, use DP with binary search).

### Approach
Sort jobs by end time. `dp[i]` = max profit using jobs `0..i`. For each job, binary-search for the latest non-overlapping job and pick the better of including or excluding.

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
from bisect import bisect_right

def jobSchedulingSimple(jobs):
    jobs.sort(key=lambda x: x[1])
    ends = [j[1] for j in jobs]
    dp = [0] * (len(jobs) + 1)
    for i, (s, e, p) in enumerate(jobs):
        k = bisect_right(ends, s)  # compatible if start >= prev_end
        dp[i+1] = max(dp[i], dp[k] + p)
    return dp[-1]

assert jobSchedulingSimple([[1,2,50],[3,5,20],[6,9,100],[2,100,200]]) == 250
assert jobSchedulingSimple([[1,3,20],[2,5,20],[3,10,100],[4,6,70],[6,9,60]]) == 150
assert jobSchedulingSimple([[1,2,1],[1,2,2],[1,2,3]]) == 3
print("All tests passed!")

### Medium 20 — Maximum Sum of Non-Overlapping Intervals

> 🏢 **Asked by:** Amazon, Google
Given intervals `[start, end, value]`, find the maximum sum of values from a set of non-overlapping intervals (intervals are open-ended: `[start, end)`).

### Approach
Sort by end. DP: `dp[i]` = best total using first `i` intervals. Binary-search for the last interval ending <= current start.

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
from bisect import bisect_right

def maxSumNonOverlap(intervals):
    intervals.sort(key=lambda x: x[1])
    ends = [iv[1] for iv in intervals]
    dp = [0] * (len(intervals) + 1)
    for i, (s, e, v) in enumerate(intervals):
        k = bisect_right(ends, s)
        dp[i+1] = max(dp[i], dp[k] + v)
    return dp[-1]

assert maxSumNonOverlap([[1,3,4],[2,5,6],[3,7,2],[5,8,5]]) == 11
assert maxSumNonOverlap([[1,2,10],[2,3,20],[3,4,30]]) == 60
assert maxSumNonOverlap([[1,5,5],[2,4,3],[3,6,2]]) == 5
print("All tests passed!")

### Medium 21 — Sweep Line: Count Active Intervals at Each Integer Point

> 🏢 **Asked by:** Amazon, Google
Given a list of `[start, end]` intervals and a range `[lo, hi]`, return a list where `result[i]` = number of intervals active at point `lo + i`.

### Approach
Build a difference array of size `hi - lo + 2`. For each interval, increment at `max(start, lo)` and decrement after `min(end, hi)`. Take prefix sums.

**Time:** O(n + (hi-lo)) | **Space:** O(hi-lo)

In [ ]:
def countActive(intervals, lo, hi):
    size = hi - lo + 1
    diff = [0] * (size + 1)
    for s, e in intervals:
        l = max(s, lo) - lo
        r = min(e, hi) - lo
        if l <= r:
            diff[l] += 1
            if r + 1 <= size:
                diff[r + 1] -= 1
    result = []
    cur = 0
    for i in range(size):
        cur += diff[i]
        result.append(cur)
    return result

assert countActive([[1,3],[2,4],[5,7]], 1, 7) == [1,2,2,1,1,1,1]
assert countActive([[1,5],[2,3]], 1, 5) == [1,2,2,1,1]
assert countActive([], 0, 3) == [0,0,0,0]
print("All tests passed!")

### Medium 22 — Minimum Platforms Required (Railway Station Classic)

> 🏢 **Asked by:** Amazon, Google
Given arrival and departure times of trains, find the minimum number of platforms required so no train waits.

### Approach
Sort arrivals and departures independently. Use two-pointer sweep: if next arrival <= next departure, a new platform is needed.

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
def minPlatforms(arrivals, departures):
    arrivals.sort()
    departures.sort()
    platforms = 1
    max_platforms = 1
    i = j = 0
    i += 1
    while i < len(arrivals) and j < len(departures):
        if arrivals[i] <= departures[j]:
            platforms += 1
            i += 1
        else:
            platforms -= 1
            j += 1
        max_platforms = max(max_platforms, platforms)
    return max_platforms

assert minPlatforms([900,940,950,1100,1500,1800],[910,1200,1120,1130,1900,2000]) == 3
assert minPlatforms([900,1100],[1000,1200]) == 1
assert minPlatforms([100,200,300],[400,500,600]) == 3
print("All tests passed!")

### Medium 23 — Overlapping Rectangles Total Area

> 🏢 **Asked by:** Amazon, Google
Given two axis-aligned rectangles defined by `(x1,y1,x2,y2)`, return the total area covered by their union.

### Approach
Area of union = area(A) + area(B) - area(intersection). The intersection rectangle has sides `[max(x1A,x1B), min(x2A,x2B)]` and `[max(y1A,y1B), min(y2A,y2B)]` if positive.

**Time:** O(1) | **Space:** O(1)

In [ ]:
def computeArea(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    intersection = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    return area_a + area_b - intersection

assert computeArea(-3,0,3,4,0,-1,9,2) == 45
assert computeArea(-2,-2,2,2,-2,-2,2,2) == 16
assert computeArea(0,0,1,1,2,2,3,3) == 2
print("All tests passed!")

### Medium 24 — Merge k Interval Lists

> 🏢 **Asked by:** Amazon, Google
Given k sorted interval lists, merge all of them into a single sorted, merged interval list with no overlaps.

### Approach
Flatten all intervals from all k lists into one list, sort by start, then apply the standard merge algorithm.

**Time:** O(N log N) where N = total intervals | **Space:** O(N)

In [ ]:
def mergeKLists(lists):
    all_intervals = [iv for lst in lists for iv in lst]
    if not all_intervals:
        return []
    all_intervals.sort()
    merged = [list(all_intervals[0])]
    for s, e in all_intervals[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return merged

assert mergeKLists([[[1,3],[5,7]],[[2,4],[6,8]],[[9,10]]]) == [[1,4],[5,8],[9,10]]
assert mergeKLists([[[1,2]],[[3,4]],[[5,6]]]) == [[1,2],[3,4],[5,6]]
assert mergeKLists([]) == []
print("All tests passed!")

### Medium 25 — Find Median from Data Stream (LC 295)

> 🏢 **Asked by:** Amazon, Google
Design a data structure that supports `addNum(int)` and `findMedian()`. `findMedian` returns the median of all numbers added so far.

### Approach
Maintain two heaps: a max-heap for the lower half and a min-heap for the upper half. Balance so their sizes differ by at most 1.

**Time:** O(log n) add, O(1) median | **Space:** O(n)

In [ ]:
import heapq

class MedianFinder:
    def __init__(self):
        self.lo = []  # max-heap (negate values)
        self.hi = []  # min-heap

    def addNum(self, num):
        heapq.heappush(self.lo, -num)
        heapq.heappush(self.hi, -heapq.heappop(self.lo))
        if len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def findMedian(self):
        if len(self.lo) > len(self.hi):
            return float(-self.lo[0])
        return (-self.lo[0] + self.hi[0]) / 2.0

mf = MedianFinder()
mf.addNum(1); mf.addNum(2)
assert mf.findMedian() == 1.5
mf.addNum(3)
assert mf.findMedian() == 2.0
mf.addNum(4)
assert mf.findMedian() == 2.5
print("All tests passed!")

### Medium 26 — Describe the Painting (LC 1943)

> 🏢 **Asked by:** Amazon, Google
Given a list of `[left, right, color]` painting strokes on a 1D canvas, return a list of `[left, right, colorSet]` describing the final appearance (a segment is one if it has no color changes within it).

### Approach
Sweep-line with a difference map: at each `left` add `color`, at `right` remove `color`. Sort all boundary points, reconstruct color sets between consecutive points.

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
from collections import defaultdict

def splitPainting(segments):
    events = defaultdict(set)
    for l, r, c in segments:
        events[l].add(('add', c))
        events[r].add(('remove', c))
    points = sorted(events)
    result = []
    active = set()
    prev = None
    for p in points:
        if prev is not None and active:
            result.append([prev, p, frozenset(active)])
        for op, c in events[p]:
            if op == 'add':
                active.add(c)
            else:
                active.discard(c)
        prev = p
    return result

res = splitPainting([[1,4,5],[4,7,7],[1,7,3]])
assert res[0][:2] == [1, 4]
assert 5 in res[0][2] and 3 in res[0][2]
assert res[1][:2] == [4, 7]
assert 7 in res[1][2] and 3 in res[1][2]
print("All tests passed!")

### Medium 27 — Amount of New Area Painted Each Day (LC 2158 simplified)

> 🏢 **Asked by:** Amazon, Google
Each day you paint `[start, end)`. Return an array where `result[i]` is the new (previously unpainted) length painted on day i.

### Approach
Maintain a set of painted segments using a sorted structure (here, a simple `painted` boolean array for clarity). For each day's range, count unpainted cells and mark them.

**Time:** O(n * maxLen) naive | **Space:** O(maxLen)

In [ ]:
def amountPainted(paint):
    # Simple approach with a painted set tracking endpoints
    # Using a workload-compression trick
    painted = {}
    result = []
    for s, e in paint:
        new_area = 0
        i = s
        while i < e:
            if i in painted:
                i = painted[i]  # jump to end of painted region
            else:
                new_area += 1
                painted[i] = i + 1
                i += 1
        # path compression
        i = s
        while i < e:
            nxt = painted[i]
            painted[i] = e
            i = nxt
        result.append(new_area)
    return result

assert amountPainted([[1,4],[4,7],[5,8]]) == [3,3,1]
assert amountPainted([[1,5],[1,5]]) == [4,0]
assert amountPainted([[1,2],[2,3],[3,4]]) == [1,1,1]
print("All tests passed!")

### Medium 28 — Minimum Interval to Include Each Query (LC 2158 full)

> 🏢 **Asked by:** Amazon, Google
Given intervals and queries, for each query find the size of the smallest interval containing that query point. Return -1 if none.

### Approach
Sort intervals and queries. Process queries in ascending order; use a min-heap keyed by interval size. Add all intervals starting <= query, pop expired ones, the heap top is the answer.

**Time:** O((n+q) log n) | **Space:** O(n+q)

In [ ]:
import heapq

def minInterval(intervals, queries):
    intervals.sort()
    indexed = sorted(enumerate(queries), key=lambda x: x[1])
    ans = [-1] * len(queries)
    heap = []  # (size, end)
    i = 0
    for idx, q in indexed:
        while i < len(intervals) and intervals[i][0] <= q:
            s, e = intervals[i]
            heapq.heappush(heap, (e - s + 1, e))
            i += 1
        while heap and heap[0][1] < q:
            heapq.heappop(heap)
        if heap:
            ans[idx] = heap[0][0]
    return ans

assert minInterval([[1,4],[2,4],[3,6],[4,4]], [2,3,4,5]) == [3,3,1,4]
assert minInterval([[2,3],[2,5],[1,8],[20,25]], [2,19,5,22]) == [2,-1,4,6]
print("All tests passed!")

### Medium 29 — Largest Rectangle in Histogram (LC 84)

> 🏢 **Asked by:** Amazon, Google
Given an array of bar heights, find the area of the largest rectangle that can be formed within the histogram.

### Approach
Use a monotone stack. Maintain indices of bars in increasing height order. When a shorter bar is encountered, pop and compute the rectangle area with the popped bar as the shortest.

**Time:** O(n) | **Space:** O(n)

In [ ]:
def largestRectangleArea(heights):
    stack = []
    max_area = 0
    heights = heights + [0]
    for i, h in enumerate(heights):
        start = i
        while stack and stack[-1][1] > h:
            idx, height = stack.pop()
            max_area = max(max_area, height * (i - idx))
            start = idx
        stack.append((start, h))
    return max_area

assert largestRectangleArea([2,1,5,6,2,3]) == 10
assert largestRectangleArea([2,4]) == 4
assert largestRectangleArea([1]) == 1
assert largestRectangleArea([1,1,1,1]) == 4
print("All tests passed!")

### Medium 30 — Count Integers in Intervals (LC 2276)

> 🏢 **Asked by:** Amazon, Google
Design a data structure that supports `add(left, right)` (adds interval [left, right]) and `count()` (returns total distinct integers covered).

### Approach
Store non-overlapping intervals in a sorted list. On `add`, merge all overlapping existing intervals with the new one, updating the count.

**Time:** O(n) per add amortized | **Space:** O(n)

In [ ]:
from sortedcontainers import SortedList

class CountIntervals:
    def __init__(self):
        self.intervals = SortedList(key=lambda x: x[0])
        self.total = 0

    def add(self, left, right):
        new_l, new_r = left, right
        to_remove = []
        # find overlapping intervals
        idx = self.intervals.bisect_left([left])
        # check left neighbor
        if idx > 0:
            l, r = self.intervals[idx-1]
            if r >= left:
                to_remove.append((l, r))
                new_l = min(new_l, l)
                new_r = max(new_r, r)
        while idx < len(self.intervals):
            l, r = self.intervals[idx]
            if l <= right:
                to_remove.append((l, r))
                new_l = min(new_l, l)
                new_r = max(new_r, r)
                idx += 1
            else:
                break
        for iv in to_remove:
            self.intervals.remove(iv)
            self.total -= iv[1] - iv[0] + 1
        self.intervals.add((new_l, new_r))
        self.total += new_r - new_l + 1

    def count(self):
        return self.total

ci = CountIntervals()
ci.add(2, 3); ci.add(7, 10)
assert ci.count() == 6
ci.add(5, 8)
assert ci.count() == 8
print("All tests passed!")

---
## Hard Problems (11–20)

### Hard 11 — Count Integers in Intervals Full (LC 2276)

> 🏢 **Asked by:** Amazon, Google
Full implementation of `CountIntervals` supporting millions of operations. Uses a sorted dict of `{start: end}` for O(n) amortized add.

### Approach
Store intervals in a `SortedDict` keyed by start. On `add(l, r)`, find and merge all intervals whose start <= r and end >= l into one. Track total with incremental updates.

**Time:** O(log n) amortized per add | **Space:** O(n)

In [ ]:
from sortedcontainers import SortedDict

class CountIntervalsFull:
    def __init__(self):
        self.sd = SortedDict()  # start -> end
        self.total = 0

    def add(self, left, right):
        sd = self.sd
        new_l, new_r = left, right
        # find leftmost interval that can overlap
        idx = sd.bisect_right(right)
        remove_keys = []
        # walk backwards to find all overlapping intervals
        i = sd.bisect_left(left) - 1
        if i >= 0:
            k = sd.keys()[i]
            if sd[k] >= left:
                new_l = min(new_l, k)
                new_r = max(new_r, sd[k])
                remove_keys.append(k)
        for k in list(sd.irange(left, right)):
            new_l = min(new_l, k)
            new_r = max(new_r, sd[k])
            remove_keys.append(k)
        for k in remove_keys:
            self.total -= sd[k] - k + 1
            del sd[k]
        sd[new_l] = new_r
        self.total += new_r - new_l + 1

    def count(self):
        return self.total

cif = CountIntervalsFull()
cif.add(2, 3); cif.add(7, 10)
assert cif.count() == 6
cif.add(5, 8)
assert cif.count() == 8
cif.add(1, 10)
assert cif.count() == 10
print("All tests passed!")

### Hard 12 — Maximum Profit in Job Scheduling (LC 1235)

> 🏢 **Asked by:** Amazon, Google, Meta
Given jobs `[startTime, endTime, profit]`, find the maximum profit from a subset of non-overlapping jobs.

### Approach
Sort by end time. `dp[i]` = max profit using jobs among first i. For each job, binary-search for last non-conflicting job index.

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
from bisect import bisect_right

def jobScheduling(startTime, endTime, profit):
    jobs = sorted(zip(startTime, endTime, profit), key=lambda x: x[1])
    ends = [j[1] for j in jobs]
    dp = [0] * (len(jobs) + 1)
    for i, (s, e, p) in enumerate(jobs):
        k = bisect_right(ends, s)
        dp[i+1] = max(dp[i], dp[k] + p)
    return dp[-1]

assert jobScheduling([1,2,3,3],[3,4,5,6],[50,10,40,70]) == 120
assert jobScheduling([1,2,3,4,6],[3,5,10,6,9],[20,20,100,70,60]) == 150
assert jobScheduling([1,1,1],[2,3,4],[5,6,4]) == 6
print("All tests passed!")

### Hard 13 — Video Stitching (LC 1024)

> 🏢 **Asked by:** Amazon, Google
Given clips `[start, end]` and a target duration `T`, find the minimum number of clips needed to cover `[0, T]`. Return -1 if impossible.

### Approach
Greedy interval cover: sort clips by start. At each step, extend coverage as far as possible using clips that begin within current coverage.

**Time:** O(n log n) | **Space:** O(1)

In [ ]:
def videoStitching(clips, time):
    clips.sort()
    count = 0
    cur_end = 0
    far = 0
    i = 0
    while cur_end < time:
        while i < len(clips) and clips[i][0] <= cur_end:
            far = max(far, clips[i][1])
            i += 1
        if far <= cur_end:
            return -1
        cur_end = far
        count += 1
    return count

assert videoStitching([[0,2],[4,6],[8,10],[1,9],[1,5],[5,9]], 10) == 3
assert videoStitching([[0,1],[1,2]], 5) == -1
assert videoStitching([[0,1],[6,8],[0,2],[5,6],[0,4],[0,3],[6,7],[1,3],[4,7],[1,4],[2,5],[2,6],[3,4],[4,5],[5,7],[6,9]], 9) == 3
print("All tests passed!")

### Hard 14 — Jump Game II as Interval Cover (LC 45)

> 🏢 **Asked by:** Amazon, Google
Given array `nums` where `nums[i]` is max jump from index `i`, return minimum jumps to reach the last index.

### Approach
Model each index as an interval `[i, i + nums[i]]`. Use the greedy interval cover approach: track `cur_end` (farthest reachable with current jumps) and `far` (global farthest). Each time `i == cur_end`, take a jump.

**Time:** O(n) | **Space:** O(1)

In [ ]:
def jump(nums):
    jumps = 0
    cur_end = 0
    far = 0
    for i in range(len(nums) - 1):
        far = max(far, i + nums[i])
        if i == cur_end:
            jumps += 1
            cur_end = far
    return jumps

assert jump([2,3,1,1,4]) == 2
assert jump([2,3,0,1,4]) == 2
assert jump([1,2,1,1,1]) == 3
assert jump([0]) == 0
assert jump([1,1,1,1]) == 3
print("All tests passed!")

### Hard 15 — Minimum Number of Intervals to Cover a Range

> 🏢 **Asked by:** Amazon, Google
Given a list of intervals and a target range `[lo, hi]`, find the minimum number of intervals needed to fully cover `[lo, hi]`. Return -1 if impossible.

### Approach
Classic greedy interval cover: sort by start. At each step greedily pick the interval starting <= current coverage that extends farthest.

**Time:** O(n log n) | **Space:** O(1)

In [ ]:
def minIntervalsCover(intervals, lo, hi):
    intervals.sort()
    count = 0
    cur = lo
    i = 0
    n = len(intervals)
    while cur < hi:
        far = cur
        while i < n and intervals[i][0] <= cur:
            far = max(far, intervals[i][1])
            i += 1
        if far == cur:
            return -1
        cur = far
        count += 1
    return count

assert minIntervalsCover([[1,3],[2,5],[4,7]], 1, 7) == 3
assert minIntervalsCover([[1,2],[3,4]], 1, 4) == -1
assert minIntervalsCover([[1,10]], 1, 10) == 1
assert minIntervalsCover([[1,3],[2,4],[3,6],[5,8]], 1, 8) == 3
print("All tests passed!")

### Hard 16 — Weighted Job Scheduling (Maximize Profit)

> 🏢 **Asked by:** Amazon, Google, Microsoft
Given jobs with `start`, `end`, and `profit`, find the maximum profit achievable by scheduling non-overlapping jobs.

### Approach
Sort by end time. For each job, binary-search for the last job that ends before the current job starts. DP recurrence: `dp[i] = max(dp[i-1], profit[i] + dp[last_compatible])`.

**Time:** O(n log n) | **Space:** O(n)

In [ ]:
from bisect import bisect_right

def weightedJobSchedule(jobs):
    # jobs: list of (start, end, profit)
    jobs.sort(key=lambda x: x[1])
    n = len(jobs)
    ends = [j[1] for j in jobs]
    dp = [0] * (n + 1)
    for i, (s, e, p) in enumerate(jobs):
        # Compatible if next job starts >= prev end
        k = bisect_right(ends, s)
        dp[i+1] = max(dp[i], dp[k] + p)
    return dp[n]

assert weightedJobSchedule([(1,3,20),(2,5,20),(3,10,100),(4,6,70),(6,9,60)]) == 150
assert weightedJobSchedule([(1,2,50),(1,3,10),(2,3,40),(2,4,25)]) == 90
assert weightedJobSchedule([(1,4,3),(2,5,4),(3,6,5)]) == 5
print("All tests passed!")

### Hard 17 — Minimum Cost to Cut a Stick Full (LC 1547)

> 🏢 **Asked by:** Amazon, Google
A stick of length `n`. Given cut positions, find the minimum total cost to make all cuts. Cost of a cut = length of the stick being cut.

### Approach
Interval DP. Add 0 and n as boundaries. `dp[i][j]` = min cost to make all cuts between position `cuts[i]` and `cuts[j]`. Try each cut `k` between `i` and `j`.

**Time:** O(m³) where m = len(cuts) | **Space:** O(m²)

In [ ]:
def minCost(n, cuts):
    cuts = sorted([0] + cuts + [n])
    m = len(cuts)
    dp = [[0] * m for _ in range(m)]
    for length in range(2, m):  # length of interval in index space
        for i in range(m - length):
            j = i + length
            dp[i][j] = float('inf')
            for k in range(i+1, j):
                cost = cuts[j] - cuts[i] + dp[i][k] + dp[k][j]
                dp[i][j] = min(dp[i][j], cost)
    return dp[0][m-1]

assert minCost(7, [1,3,4,5]) == 16
assert minCost(9, [5,6,1,4,2]) == 22
assert minCost(4, [2]) == 4
print("All tests passed!")

### Hard 18 — Strange Printer II (LC 1591)

> 🏢 **Asked by:** Google, Amazon
A printer can print a rectangle of one color per operation, overwriting previous colors. Given a target color grid, determine if it can be produced. Return True if the configuration is achievable.

### Approach
For each color, find its bounding rectangle. If any other color appears inside that bounding box, that color must be printed after the current one (topological ordering). Build a DAG and check for cycles.

**Time:** O(C * m * n) | **Space:** O(C²)

In [ ]:
def isPrintable(targetGrid):
    from collections import defaultdict, deque
    m, n = len(targetGrid), len(targetGrid[0])
    bounds = {}
    for i in range(m):
        for j in range(n):
            c = targetGrid[i][j]
            if c not in bounds:
                bounds[c] = [i, j, i, j]
            else:
                bounds[c][0] = min(bounds[c][0], i)
                bounds[c][1] = min(bounds[c][1], j)
                bounds[c][2] = max(bounds[c][2], i)
                bounds[c][3] = max(bounds[c][3], j)
    graph = defaultdict(set)
    indegree = defaultdict(int)
    colors = set(bounds)
    for c in colors:
        r1, c1, r2, c2 = bounds[c]
        for i in range(r1, r2+1):
            for j in range(c1, c2+1):
                other = targetGrid[i][j]
                if other != c and other not in graph[c]:
                    graph[c].add(other)
                    indegree[other] = indegree.get(other, 0) + 1
    for c in colors:
        if c not in indegree:
            indegree[c] = 0
    q = deque(c for c in colors if indegree[c] == 0)
    visited = 0
    while q:
        c = q.popleft()
        visited += 1
        for nb in graph[c]:
            indegree[nb] -= 1
            if indegree[nb] == 0:
                q.append(nb)
    return visited == len(colors)

assert isPrintable([[1,1,1,1],[1,2,2,1],[1,2,2,1],[1,1,1,1]]) == True
assert isPrintable([[1,1,1],[3,1,3]]) == False
assert isPrintable([[1,2],[2,1]]) == False
print("All tests passed!")

### Hard 19 — Minimum Difficulty of a Job Schedule (LC 1335)

> 🏢 **Asked by:** Amazon, Google
Schedule `n` jobs over `d` days (at least 1 job per day). Daily difficulty = max job difficulty that day. Minimize total difficulty.

### Approach
DP: `dp[i][j]` = min difficulty scheduling first `j` jobs in `i` days. Transition: try all split points for the last day.

**Time:** O(d * n²) | **Space:** O(d * n)

In [ ]:
def minDifficulty(jobDifficulty, d):
    n = len(jobDifficulty)
    if n < d:
        return -1
    INF = float('inf')
    # dp[i][j] = min difficulty for first j jobs in i days
    dp = [[INF] * (n+1) for _ in range(d+1)]
    dp[0][0] = 0
    for day in range(1, d+1):
        for j in range(day, n - d + day + 1):
            max_d = 0
            for k in range(j, day-1, -1):
                max_d = max(max_d, jobDifficulty[k-1])
                if dp[day-1][k-1] < INF:
                    dp[day][j] = min(dp[day][j], dp[day-1][k-1] + max_d)
    return dp[d][n] if dp[d][n] < INF else -1

assert minDifficulty([6,5,4,3,2,1], 2) == 7
assert minDifficulty([9,9,9], 4) == -1
assert minDifficulty([1,1,1], 3) == 3
assert minDifficulty([7,1,7,1,7,1], 3) == 15
print("All tests passed!")

### Hard 20 — Zuma Game with Interval DP (LC 488)

> 🏢 **Asked by:** Google, Amazon
Given a board string and a hand of balls, find the minimum number of hand balls needed to clear the board. Return -1 if impossible.

### Approach
Use memoized recursion. Compress consecutive identical balls into groups. For each group, try inserting hand balls to make a run of 3+, then recurse on the remaining board and hand.

**Time:** Exponential worst case, manageable with memoization | **Space:** O(state space)

In [ ]:
from functools import lru_cache

def findMinStep(board, hand):
    from collections import Counter
    hand_count = Counter(hand)

    def compress(s):
        # Remove groups of 3+ consecutive same chars
        changed = True
        while changed:
            changed = False
            i = 0
            ns = []
            while i < len(s):
                j = i
                while j < len(s) and s[j] == s[i]:
                    j += 1
                if j - i >= 3:
                    changed = True
                else:
                    ns.extend(s[i:j])
                i = j
            s = ns
        return tuple(s)

    @lru_cache(maxsize=None)
    def dp(board_t, hand_t):
        board_t = compress(list(board_t))
        if not board_t:
            return 0
        hand_c = Counter(dict(zip(hand_t[::2], hand_t[1::2])))
        res = float('inf')
        i = 0
        while i < len(board_t):
            j = i
            while j < len(board_t) and board_t[j] == board_t[i]:
                j += 1
            color = board_t[i]
            need = 3 - (j - i)
            if hand_c[color] >= need:
                hand_c[color] -= need
                new_hand = tuple(sorted((c, n) for c, n in hand_c.items() if n > 0))
                new_hand_flat = tuple(x for pair in new_hand for x in pair)
                sub = dp(board_t[:i] + board_t[j:], new_hand_flat)
                if sub < float('inf'):
                    res = min(res, need + sub)
                hand_c[color] += need
            i = j
        return res

    hand_flat = tuple(x for pair in sorted(hand_count.items()) for x in pair)
    ans = dp(tuple(board), hand_flat)
    return ans if ans < float('inf') else -1

assert findMinStep('WRRBBW', 'RB') == -1
assert findMinStep('WWRRBBWW', 'WRBRW') == 2
assert findMinStep('G', 'GGGGG') == 2
print("All tests passed!")